In [3]:
from calendar import calendar
import pandas as pd
import json
from pathlib import Path
from sqlalchemy import create_engine
import timeit
# from sqlalchemy.orm import Session,sessionmaker
# Using SQLAlchemy to connect to the Database

from sqlalchemy import create_engine,MetaData,event
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

# from .utils.log_helper import *


engine = create_engine("", echo=False)


Session = sessionmaker(autocommit=False, autoflush=False, bind=engine)

session = Session()

Base = declarative_base(metadata=MetaData(schema='metro_api_dev'))

def get_db():
    db = Session()
    try:
        print('Connected to the database')
        yield db
    finally:
        db.close()

list_of_gtfs_static_files = ["stop_times"]

def update_gtfs_static_files():
    for file in list_of_gtfs_static_files:
        print(f"Updating {file}")
        process_start = timeit.default_timer()
        # update_gtfs_realtime_data()
        dataframe_to_delete = []
        bus_file_path = "../../appdata/gtfs-static/gtfs_bus/" + file + '.txt'
        rail_file_path = "../../appdata/gtfs-static/gtfs_rail/" + file + '.txt'
        temp_df_bus = pd.read_csv(bus_file_path)
        temp_df_bus['agency_id'] = 'LACMTA'
        temp_df_rail = pd.read_csv(rail_file_path)
        temp_df_rail['agency_id'] = 'LACMTA_Rail'
        combined_temp_df = pd.concat([temp_df_bus, temp_df_rail])
        combined_temp_df.to_sql(file+"_new",engine,index=False,if_exists="replace",schema='metro_api')
        dataframe_to_delete.append(combined_temp_df)
        del dataframe_to_delete
        process_end = timeit.default_timer()
        print('Updating took {} seconds'.format(process_end - process_start))

update_gtfs_static_files()

Updating stop_times


<ipython-input-3-2a810c418936>:55: DtypeWarning: Columns (9,11) have mixed types.Specify dtype option on import or set low_memory=False.
  update_gtfs_static_files()
